In [3]:
import os
import torch
import clip
import numpy
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torch.nn.utils as nn_utils  # For gradient norm computation
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import torch.nn.functional as F
import matplotlib.pyplot as plt
from diffusers import UNet2DConditionModel, StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor
import torch


In [4]:
class ImageTextDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        super().__init__()
        self.data_dir = data_dir
        self.transform = transform

        # Get all .jpg files and check for corresponding non-empty .txt files
        self.image_filenames = sorted([f for f in os.listdir(data_dir) if f.endswith(".jpg")])
        
        # Filter out pairs with empty or missing captions
        self.image_filenames = [
            f for f in self.image_filenames 
            if os.path.exists(os.path.join(data_dir, f.replace(".jpg", ".txt"))) 
            and os.path.getsize(os.path.join(data_dir, f.replace(".jpg", ".txt"))) > 0
        ]
        
        print(f"Total valid image-caption pairs: {len(self.image_filenames)}")

    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]
        img_path = os.path.join(self.data_dir, img_name)
    
        # Load image
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Error loading image {img_name}: {e}")
    
        # Load corresponding caption
        caption_file = img_name.replace(".jpg", ".txt")
        caption_path = os.path.join(self.data_dir, caption_file)
    
        try:
            with open(caption_path, "r") as file:
                caption = file.read().strip()
        except Exception as e:
            raise RuntimeError(f"Error loading caption {caption_file}: {e}")
    
        if self.transform:
            image = self.transform(image)
    
        return image, caption


# Load the pre-trained CLIP model
device = "cuda" 
model, preprocess = clip.load("ViT-L/14", device=device)

# Paths to your image and caption directories
data_dir = r"C:\Users\Osama\Desktop\Master\DeepLearning\deep-machine-learning\SSY340---Deep-Learning---Text-To-Image-Generation\images2\images2"

# Define image transformations (resizing, normalizing as per CLIP's expected input)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Ensure the right size for the model
    transforms.ToTensor(),
])

# Create dataset and dataloader with optimized loading
small_dataset = ImageTextDataset(data_dir=data_dir, transform=transform)
# Assuming your ImageTextDataset is correctly defined and data loaded
#batch_size = 16
#dataloader = DataLoader(small_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

#total_batches = len(dataloader)
#print(f"Total number of batches: {total_batches}")


# Set your desired split ratio
train_ratio = 0.8  # 80% for training
val_ratio = 0.2    # 20% for validation

# Calculate the sizes of the train and validation sets
dataset_size = len(small_dataset)
train_size = int(train_ratio * dataset_size)
val_size = dataset_size - train_size  # Remainder for validation

# Split the dataset into train and validation sets
train_dataset, val_dataset = random_split(small_dataset, [train_size, val_size])

# Create DataLoaders for both the training and validation sets
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)


Total valid image-caption pairs: 24445


In [10]:
import os
import random
import shutil

def split_dataset(source_folder, train_folder, test_folder, train_percentage=0.8):
    # Create directories if they don't exist
    if not os.path.exists(train_folder):
        os.makedirs(train_folder)
    if not os.path.exists(test_folder):
        os.makedirs(test_folder)
    
    # Get a list of all files in the source folder
    all_files = os.listdir(source_folder)
    
    # Shuffle the files randomly
    #random.shuffle(all_files)
    
    # Calculate the number of training files
    train_size = int(len(all_files) * train_percentage)
    
    # Split the files into train and test sets
    train_files = all_files[:train_size]
    test_files = all_files[train_size:]
    
    # Move files to train and test folders
    for file_name in train_files:
        src_path = os.path.join(source_folder, file_name)
        dest_path = os.path.join(train_folder, file_name)
        shutil.move(src_path, dest_path)
    
    for file_name in test_files:
        src_path = os.path.join(source_folder, file_name)
        dest_path = os.path.join(test_folder, file_name)
        shutil.move(src_path, dest_path)
    
    print(f"Moved {len(train_files)} files to {train_folder}")
    print(f"Moved {len(test_files)} files to {test_folder}")

# Example usage
source_folder = r"C:\Users\Osama\Desktop\Master\DeepLearning\deep-machine-learning\SSY340---Deep-Learning---Text-To-Image-Generation\images_small"

train_folder = 'train'
test_folder = 'test'
train_percentage = 0.8  # 80% for training, 20% for testing

split_dataset(source_folder, train_folder, test_folder, train_percentage)


Moved 9383 files to train
Moved 2346 files to test
